In [ ]:
# Feel free to use this as a starter for now. Am building upon it one hour a day

# Basic Imports

In [ ]:
import numpy
import pandas
import matplotlib
import tqdm
df = pandas.read_csv('/kaggle/input/tmdb-movies-dataset-2023-930k-movies/TMDB_movie_dataset_v11.csv')

In [ ]:
df.head()

In [ ]:
# Check for na values over 2d 
print(f'Presence of NA values in the dataset : {df.isna().any().any()}')
print(f'Columns : {list(df.columns)}')

# Cleaning the dataset

We follow the format shown below to provide a cleaner format for accessing the datasets
The changes that we did to the data will be highlighted here.

- Bronze tier : df
- Silver tier :
    - released_movies
    - rumored_movies
    - planned_movies
    - inproduction_movies
    - postproduction_movies
    - cancelled_movies
- Gold tier
    - TODO
    
![Delta lake concept](https://delta.io/static/delta-hp-hero-bottom-46084c40468376aaecdedc066291e2d8.png)

# NaN values detection

The dataset contains everthing in strings hence it becomes our responsibility to detect NaN values

- There are released movies with 0 revenue which sounds really unlikely
- There are movies that even after released have 0 runtime
- There are movies with no IMDB index ( apparantly some of them are adult videos)

In [ ]:
# No imdb index
df[df['imdb_id'].isna()] # 382247

In [ ]:
df['status'].unique()

In [ ]:
# '1500-01-01'
rd = pandas.to_datetime(df['release_date'], format='%Y-%m-%d', errors = 'coerce')
imaginary_movies = df[rd.isna()]
no_imag_df = df.drop(imaginary_movies.index)
# Checking for round hay garden scene ! (The oldest movie : https://www.oldest.org/entertainment/movies/)
no_imag_df['release_date'] = pandas.to_datetime(no_imag_df['release_date'], format='%Y-%m-%d')

import datetime
no_imag_df[no_imag_df['release_date'] < numpy.datetime64('1881')].__len__()

**Note** : we will consider the movies after the date 1891-01-01 on part that the world had advanced in filming technologies before the Great War however film had not propogated far by then and things before then were heavily reliant on non-celluloid films

In [ ]:
historic_films = no_imag_df[no_imag_df['release_date'] < numpy.datetime64('1891')]
no_hist_df = no_imag_df.drop(historic_films.index)

joke_movies = no_hist_df[no_hist_df['release_date'] > numpy.datetime64('2030')]

**Note** These are all joke listings of a movie hence we can safely remove these

In [ ]:
no_hist_fut_df = no_hist_df.drop(joke_movies.index)

In [ ]:
released_movies = no_hist_fut_df[no_hist_fut_df['status'] == 'Released']
rumored_movies = no_hist_fut_df[no_hist_fut_df['status'] =='Rumored']
planned_movies = no_hist_fut_df[no_hist_fut_df['status'] =='Planned']
inproduction_movies = no_hist_fut_df[no_hist_fut_df['status'] =='In Production']
postproduction_movies = no_hist_fut_df[no_hist_fut_df['status'] =='Post Production']
cancelled_movies = no_hist_fut_df[no_hist_fut_df['status'] =='Canceled']

# Types of datasets constructed

1. historic_movies
2. planeed_movies
3. joke_movies
4. cancelled_movies
5. released_movies
6. rumored_movies
7. inproduction_movies
8. postproduction_movies
9. cancelled_movies

In [ ]:
print(f'Length of Historic movies        : {historic_films.__len__()}')
print(f'Length of Future movies          : {planned_movies.__len__()}')
print(f'Length of Joke movies            : {joke_movies.__len__()}')
print(f'Length of Cancelled movies       : {cancelled_movies.__len__()}')
print(f'Length of Released movies        : {released_movies.__len__()}')
print(f'Length of Rumored movies         : {rumored_movies.__len__()}')
print(f'Length of Inproduction movies    : {inproduction_movies.__len__()}')
print(f'Length of Post production movies : {postproduction_movies.__len__()}')
print(f'Length of Cancelled movies       : {cancelled_movies.__len__()}')

# Preliminary analysis

In [ ]:
numerical_columns : list[str] = ['vote_average', 'vote_count', 'revenue', 'runtime', 'budget', 'popularity']

# fig, axs = matplotlib.pyplot.subplots(len(numerical_columns), len(numerical_columns))
# for idx, col1 in enumerate(numerical_columns):
#     for idx2, col2 in enumerate(numerical_columns):
#         if (idx == idx2):
#             axs[idx, idx2].hist(df[col1])
#         if (idx > idx2):
#             axs[idx, idx2].scatter(x = df[col1], y = df[col2], alpha = 0.5)

_ = pandas.plotting.scatter_matrix(released_movies[numerical_columns], figsize = (10,10), alpha = 0.3)

In [ ]:
# Correlations

corr = released_movies[numerical_columns].corr()
corr.style.background_gradient(cmap='coolwarm')

In [ ]:
# This is an attempt to heirarchicaly cluster the above dataset
import scipy

# Why is there no documentation for this >>>>?????
pairwise_dist_mat = scipy.spatial.distance.pdist(corr)
linkage_matrix = scipy.cluster.hierarchy.linkage(pairwise_dist_mat, 'complete')
idx = scipy.cluster.hierarchy.fcluster(linkage_matrix, 0.5 * pairwise_dist_mat.max(), 'distance')

columns = [released_movies[numerical_columns].columns.tolist()[i] for i in list((numpy.argsort(idx)))]
corr = released_movies[numerical_columns][columns].corr()
corr.style.background_gradient(cmap='coolwarm')

# Important ideas

1. Predict movie ratings based on features such as revenue, popularity, genre, and runtime.
2. Identify trends in movie release dates and analyze their impact on revenue.
3. Analyze the relationship between budget, revenue, and popularity to determine factors that contribute to a movie's success.
4. Build a recommendation system that suggests similar movies based on genres, production companies, and language.
5. Perform sentiment analysis on movie reviews to understand audience reactions.
6. Explore the impact of movie genres on popularity and revenue.
7. Investigate the correlation between runtime and audience engagement.
8. Identify successful production companies and analyze their strategies.
9. Utilize natural language processing techniques to extract meaningful insights from movie overviews.
10. Visualize movie popularity over time and identify popular genres in different periods.

# Predict movie ratings

In [ ]:
# We build the model according to the ratings given first and see if other features enhance the accuracy of the model

# We will go ahead with the silver tier data to show some workings but there are some discrepencies in it as mentioned above
dataset = released_movies[['revenue', 'popularity', 'genres', 'runtime', 'vote_average']].copy()

The genre column seems like the hardest to sift through

In [ ]:
import sklearn.preprocessing
import sklearn.linear_model

regressor = sklearn.linear_model.LinearRegression()

genres = dataset['genres']
genres.str.split(',')

In [ ]:
ag = genres.str.replace(' ', '').str.split(',')
all_genres = set()
for x in ag:
    if (type(x) == type([0])):
        all_genres = all_genres.union(set(x))
    
print(all_genres)

In [ ]:
for genre in all_genres:    
    dataset.loc[:,genre] = ag.apply(lambda x : genre in x if type(x) == type([]) else False)
dataset_ = dataset.drop('genres', axis = 1)
dataset_.head()

## Data types

There are positive revenue movies (>0) amounting to 17885 of the released movies and Zero revenue movies amounting to 830191 movies.

These zero revenue movies amount to some movies where the budget is unknown, or maybe some movies where it was rehashed into another product. There might be another case where the movie might be of pornographic nature where the budget is not known or disclosed.

For our case we build 2 models everytime positive (>0) and all (>=0) models

In [ ]:
# Revenue zero movies
# df.reindex(dataset_[dataset_.revenue == 0].index) #830191

# Positive revenue movies
# df.reindex(dataset_[dataset_.revenue > 0].index) # 17885

In [ ]:
import sklearn.model_selection
import sklearn.preprocessing


X_all = dataset_.drop('revenue', axis = 1)
y_all = dataset_['revenue']
scaler_all = sklearn.preprocessing.StandardScaler().fit(X_all)
X_all_scaled = scaler_all.transform(X_all)

positive_dataset = dataset_.reindex(dataset_[dataset_.revenue > 0].index)
X_positive = positive_dataset.drop('revenue', axis = 1)
y_positive = positive_dataset['revenue']
scaler_positive = sklearn.preprocessing.StandardScaler().fit(X_positive)
X_positive_scaled = scaler_positive.transform(X_positive)

X_train_all, X_test_all, y_train_all, y_test_all = sklearn.model_selection.train_test_split(
    X_all_scaled, y_all,
    test_size = 0.3,
    train_size = None,
    random_state = 42,
    shuffle = True,
    stratify = None
)

X_train_positive, X_test_positive, y_train_positive, y_test_positive = sklearn.model_selection.train_test_split(
    X_positive_scaled, y_positive,
    test_size = 0.3,
    train_size = None,
    random_state = 42,
    shuffle = True,
    stratify = None
)

## Linear Models

Reference : [Scikit-learn Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)

In [ ]:
import sklearn.linear_model

all_regressor = sklearn.linear_model.LinearRegression()
positive_regressor = sklearn.linear_model.LinearRegression()

all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)


# print(f'Coefficients for all released movies             : {all_regressor.coef_}')
print(f'R2 Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
# print(f'Coefficients for positive released movies        : {positive_regressor.coef_}')
print(f'R2 Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')

In [ ]:
fig, axs = matplotlib.pyplot.subplots(1,2, figsize = (10,10), sharey = True)
axs[0].barh(numpy.arange(len(all_regressor.coef_)), all_regressor.coef_)
_ = axs[0].set_yticks(numpy.arange(len(all_regressor.coef_)), labels=X_all.columns)
_ = axs[0].set(title = f'All Linear model with {all_regressor.score(X_test_all, y_test_all):.2f} R^2')

axs[1].barh(numpy.arange(len(positive_regressor.coef_)), positive_regressor.coef_)
_ = axs[1].set_yticks(numpy.arange(len(positive_regressor.coef_)), labels=X_positive.columns)
_ = axs[1].set(title = f'Positive Linear model with {positive_regressor.score(X_test_positive, y_test_positive):.2f} R^2')


**Note** Even accounting to non-negative regression on values the accuracy only amounts to a decrease in Linear Regression

In [ ]:
all_regressor = sklearn.linear_model.LinearRegression(positive = True)
positive_regressor = sklearn.linear_model.LinearRegression(positive = True)

all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'R2 Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'R2 Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')

fig, axs = matplotlib.pyplot.subplots(1,2, figsize = (10,10), sharey = True)
axs[0].barh(numpy.arange(len(all_regressor.coef_)), all_regressor.coef_)
_ = axs[0].set_yticks(numpy.arange(len(all_regressor.coef_)), labels=X_all.columns)
_ = axs[0].set(title = f'All Linear model with {all_regressor.score(X_test_all, y_test_all):.2f} R^2')

axs[1].barh(numpy.arange(len(positive_regressor.coef_)), positive_regressor.coef_)
_ = axs[1].set_yticks(numpy.arange(len(positive_regressor.coef_)), labels=X_positive.columns)
_ = axs[1].set(title = f'Positive Linear model with {positive_regressor.score(X_test_positive, y_test_positive):.2f} R^2')

## Ridge Regression


Even though we understand that choosing a linear model only amounts to 12% accuracy on positive released dataset and no more improvements can be done with linear models. We shall work on all of them just to find some relevnacy or improvements in the dataset.

Checked on each of them all of them provided better results for positive cases around alpha = $10^3$ - $10^4$ upto 16% accuracy. For solvers 'sag' and 'saga' taking too much time to evaluate

In [ ]:
# solver{‘auto’, ‘svd’, ‘cholesky’, ‘lsqr’, ‘sparse_cg’, ‘sag’, ‘saga’, ‘lbfgs’}, default=’auto’

alpha_coefs = 10**numpy.arange(-4,7,0.5)

all_regressor_list = [sklearn.linear_model.Ridge(alpha=alpha, fit_intercept=True, copy_X=True, max_iter=None, tol=0.0001, solver='auto', positive=False, random_state=None) for alpha in alpha_coefs]
positive_regressor_list = [sklearn.linear_model.Ridge(alpha=alpha, fit_intercept=True, copy_X=True, max_iter=None, tol=0.0001, solver='auto', positive=False, random_state=None) for alpha in alpha_coefs]

_ = [all_regressor.fit(X_train_all, y_train_all) for all_regressor in all_regressor_list]
_ = [positive_regressor.fit(X_train_positive, y_train_positive) for positive_regressor in positive_regressor_list]

# _ = [print(f'Alpha : {alpha:e} Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all) * 100:.2f}%') for alpha, all_regressor in zip(alpha_coefs, all_regressor_list)]
# _ = [print(f'Alpha : {alpha:e} Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive) * 100:.2f}%') for alpha, positive_regressor in zip(alpha_coefs,positive_regressor_list)]

all_reg_best_score = 0.0
all_reg_best_alpha = 0.0
positive_reg_best_score= 0.0
positive_reg_best_alpha= 0.0

# for alpha,regressor in zip(alpha_coefs, all_regressor_list):
#     if (regressor.score(X_test_all, y_test_all) > all_reg_best_score):
#         all_reg_best_alpha = alpha
#         all_reg_best_score = regressor.score(X_test_all, y_test_all)

# for alpha,regressor in zip(alpha_coefs, positive_regressor_list):
#     if (regressor.score(X_test_all, y_test_all) > positive_reg_best_score):
#         positive_reg_best_alpha = alpha
#         positive_reg_best_score = regressor.score(X_test_all, y_test_all)


fig, axs = matplotlib.pyplot.subplots(2,1, figsize = (10,10), sharex = True)
axs[0].semilogx(alpha_coefs, [reg.score(X_test_all, y_test_all) for reg in all_regressor_list])
_ = axs[0].set(title = f'Ridge Regression All Lasso model')

axs[1].semilogx(alpha_coefs, [reg.score(X_test_positive, y_test_positive) for reg in positive_regressor_list])
_ = axs[1].set(title = f'Ridge Regression Positive Lasso model')

## Lasso and ElasticNet

Since ElasticNet covers the case for Lasso, we shall only consider ElasticNet

0.1618 is the best $R^2$ we could produce

In [ ]:
# This is a hoarder of time
# Comment this out in case you want faster processing

import itertools

alpha_coefs = 10**numpy.arange(-1,3,0.5)
l1_ratios = numpy.arange(0,1,0.1)

all_regressor_list = [
    sklearn.linear_model.ElasticNet(
        alpha=alpha, 
        l1_ratio=l1_ratio, 
        fit_intercept=True, 
        precompute=False, 
        max_iter=1000, 
        copy_X=True, 
        tol=0.0001, 
        warm_start=False, 
        positive=False, 
        random_state=None, 
        selection='cyclic'
    ) for alpha, l1_ratio in itertools.product(alpha_coefs, l1_ratios)
]
positive_regressor_list = [
    sklearn.linear_model.ElasticNet(
        alpha=alpha, 
        l1_ratio=l1_ratio, 
        fit_intercept=True, 
        precompute=False, 
        max_iter=1000, 
        copy_X=True, 
        tol=0.0001, 
        warm_start=False, 
        positive=False, 
        random_state=None, 
        selection='cyclic'
    ) for alpha, l1_ratio in itertools.product(alpha_coefs, l1_ratios)
]



# _ = [print(f'Alpha : {alpha:e}, l1_ratio : {l1_ratio:e} Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all) * 100:.2f}%') for (alpha, l1_ratio), all_regressor in zip(itertools.product(alpha_coefs, l1_ratios), all_regressor_list)]
# _ = [print(f'Alpha : {alpha:e}, l1_ratio : {l1_ratio:e} Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive) * 100:.2f}%') for (alpha, l1_ratio), positive_regressor in zip(itertools.product(alpha_coefs, l1_ratios),positive_regressor_list)]

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _ = [all_regressor.fit(X_train_all, y_train_all) for all_regressor in all_regressor_list]
    _ = [positive_regressor.fit(X_train_positive, y_train_positive) for positive_regressor in positive_regressor_list]

In [ ]:
all_elasticnet_regressors = pandas.DataFrame(numpy.reshape(numpy.array([ar.score(X_test_all, y_test_all) for ar in all_regressor_list]), (alpha_coefs.__len__(), l1_ratios.__len__())), columns = l1_ratios, index = alpha_coefs)
all_elasticnet_regressors.style.background_gradient(cmap='coolwarm')

In [ ]:
positive_elasticnet_regressors = pandas.DataFrame(numpy.reshape(numpy.array([pr.score(X_test_positive, y_test_positive) for pr in positive_regressor_list]), (alpha_coefs.__len__(), l1_ratios.__len__())), columns = l1_ratios, index = alpha_coefs)
positive_elasticnet_regressors.style.background_gradient(cmap='coolwarm')

## Bayesian Regression

There are fail cases for positive movies for this case. Will have to investigate on it.

In [ ]:
# Two models 
# Bayesian Ridge Regressor
# Automatic Relevance Determination

# Bayesian  Ridge Regression


all_regressor = sklearn.linear_model.BayesianRidge()
positive_regressor = sklearn.linear_model.BayesianRidge()

all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'R2 Bayesian Ridge Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'R2 Bayesian Ridge Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')

all_regressor = sklearn.linear_model.ARDRegression()
positive_regressor = sklearn.linear_model.ARDRegression()

all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'R2 ARD Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'R2 ARD Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive) :.2f}')

## SGD Regressor

Increase in $R^2$ with upto 0.17 (least 0.08 seen). There might be improvements with make_pipeline

In [ ]:
all_regressor = sklearn.linear_model.SGDRegressor(
    loss='squared_error', 
    penalty=None, 
    alpha=0.0001, 
    l1_ratio=0.15, 
    fit_intercept=True, 
    max_iter=1000, 
    tol=0.001, 
    shuffle=True, 
    verbose=0, 
    epsilon=0.1, 
    random_state=None, 
    learning_rate='invscaling', 
    eta0=0.01, 
    power_t=0.25, 
    early_stopping=False, 
    validation_fraction=0.1, 
    n_iter_no_change=5, 
    warm_start=False, 
    average=False
)

positive_regressor = sklearn.linear_model.SGDRegressor(
    loss='squared_error', 
    penalty=None, 
    alpha=0.0001, 
    l1_ratio=0.15, 
    fit_intercept=True, 
    max_iter=1000, 
    tol=0.001, 
    shuffle=True, 
    verbose=0, 
    epsilon=0.1, 
    random_state=None, 
    learning_rate='invscaling', 
    eta0=0.01, 
    power_t=0.25, 
    early_stopping=False, 
    validation_fraction=0.1, 
    n_iter_no_change=5, 
    warm_start=False, 
    average=False
)

all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'SGD Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'SGD Regressor Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')


## Decision Trees


In [ ]:
import sklearn.tree

all_regressor = sklearn.tree.DecisionTreeRegressor(
    criterion='squared_error', 
    splitter='best', 
    max_depth=None, 
    min_samples_split=2, 
    min_samples_leaf=1, 
    min_weight_fraction_leaf=0.0, 
    max_features=None, 
    random_state=None, 
    max_leaf_nodes=None, 
    min_impurity_decrease=0.0, 
    ccp_alpha=0.0
)

positive_regressor = sklearn.tree.DecisionTreeRegressor(
    criterion='squared_error', 
    splitter='best', 
    max_depth=None, 
    min_samples_split=2, 
    min_samples_leaf=1, 
    min_weight_fraction_leaf=0.0, 
    max_features=None, 
    random_state=None, 
    max_leaf_nodes=None, 
    min_impurity_decrease=0.0, 
    ccp_alpha=0.0
)


all_regressor.fit(X_train_all, y_train_all)
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'R2 Decision Tree Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'R2 Decision Tree Regressor Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')


## MLP Regressor

This is a single layered perceptron with basic regressor. Immediately we see an increase in R^2 score from MLP Regressor For a $k=98$ we obtain R^2 scores with adaptive learning rate (starts with 10 and divides by 5 every 3 times the loss does not decrease), we obtain $R^2$ of 0.44 for all and 0.47 for positive. Which is pretty good.

![MLP perceptron](https://scikit-learn.org/stable/_images/multilayerperceptron_network.png)

In [ ]:
import sklearn.neural_network

all_regressor = sklearn.neural_network.MLPRegressor(
    hidden_layer_sizes=(100,), 
    activation='relu', 
    solver='adam', 
    alpha=0.0001, 
    batch_size=1024, 
    learning_rate='adaptive', 
    learning_rate_init=10,
    power_t=0.5, 
    max_iter=200, 
    shuffle=True, 
    random_state=None, 
    tol=0.0001, 
    verbose=False, 
    warm_start=False, 
    momentum=0.9, 
    nesterovs_momentum=True, 
    early_stopping=False, 
    validation_fraction=0.1, 
    beta_1=0.9, 
    beta_2=0.999, 
    epsilon=1e-08, 
    n_iter_no_change=20, 
    max_fun=15000
)

print('All')
all_regressor.fit(X_train_all, y_train_all)

In [ ]:
positive_regressor = sklearn.neural_network.MLPRegressor(
    hidden_layer_sizes=(100,), 
    activation='relu', 
    solver='adam', 
    alpha=0.0001, 
    batch_size=64, 
    learning_rate='adaptive', 
    learning_rate_init=1,
    power_t=0.5, 
    max_iter=10000, 
    shuffle=True, 
    random_state=None, 
    tol=0.0001, 
    verbose=False, 
    warm_start=False, 
    momentum=0.9, 
    nesterovs_momentum=True, 
    early_stopping=False, 
    validation_fraction=0.1, 
    beta_1=0.9, 
    beta_2=0.999, 
    epsilon=1e-08, 
    n_iter_no_change=100, 
    max_fun=15000
)

print('Positive')
positive_regressor.fit(X_train_positive, y_train_positive)

print(f'R2 Decision Tree Regressor accounting for all released movies     : {all_regressor.score(X_test_all, y_test_all):.2f}')
print(f'R2 Decision Tree Regressor Regressor accounting for postive released movies : {positive_regressor.score(X_test_positive, y_test_positive):.2f}')


In [ ]:
matplotlib.pyplot.plot(range(1, len(positive_regressor.loss_curve_)+1),positive_regressor.loss_curve_,
                      label = "Positive")
matplotlib.pyplot.legend()

In [ ]:
matplotlib.pyplot.plot(range(1, len(all_regressor.loss_curve_)+1),all_regressor.loss_curve_,
                      label = "All")
matplotlib.pyplot.title(r"MLP with R^2  All regressor")

*We shall experiment with the positive case to see effects of batch sizes and continued searching for parameters*

In [ ]:
# def model_get(batch_size:int, alpha:float) -> sklearn.neural_network.MLPRegressor:
#     """Returns a NN with defined batch_size and alpha"""

#     return sklearn.neural_network.MLPRegressor(
#         hidden_layer_sizes=(100,), 
#         activation='relu', 
#         solver='adam', 
#         alpha=alpha, 
#         batch_size=batch_size, 
#         learning_rate='adaptive', 
#         learning_rate_init=1,
#         power_t=0.5, 
#         max_iter=1000, 
#         shuffle=True, 
#         random_state=None, 
#         tol=0.0001, 
#         verbose=False, 
#         warm_start=False, 
#         momentum=0.9, 
#         nesterovs_momentum=True, 
#         early_stopping=False, 
#         validation_fraction=0.1, 
#         beta_1=0.9, 
#         beta_2=0.999, 
#         epsilon=1e-08, 
#         n_iter_no_change=100, 
#         max_fun=15000
#     )


# alpha_sizes = 10**numpy.arange(-5,1,0.5)
# batch_sizes = 2**numpy.arange(8,10)

# batches = list(itertools.product(alpha_sizes, batch_sizes))
# models : list[sklearn.neural_network.MLPRegressor] = []

# for alpha,batch_size in tqdm.tqdm(batches):
#     models.append(model_get(batch_size = batch_size, alpha = alpha))
#     models[-1].fit(X_train_positive, y_train_positive)